### End-to-End E-Commerce Data Pipeline (Databricks & PySpark)

### Data Ingestion & Source
Dataset source: [E-Commerce Dataset on Kaggle](https://www.kaggle.com/datasets/carrie1/ecommerce-data)
![image_1776278536473.png](./image_1776278536473.png "image_1776278536473.png")

### Architecture Diagram
```Architecture Diagram
[ Raw Data (.csv) ]
       |
       v
+------------------+
|   BRONZE LAYER   | <-- Ingestion (Raw Data)
+------------------+
       |
       v
+------------------+
|   SILVER LAYER   | <-- Transformation & Cleaning
+------------------+
       |
       v
+------------------+
|    GOLD LAYER    | <-- Business Logic & Analytics
+------------------+
```

**Data Ingestion (Bronze Layer)**
Load raw data into Databricks and store in Delta format. Handle schema
inference, evolution, and corrupt records.

In [0]:
# Reading our table 'data' and save it in Bronze 
df = spark.table("workspace.default.data")
df.write.format("delta").mode("overwrite").saveAsTable("default.bronze_sales")

print("Bronze Layer created")
display(spark.table("default.bronze_sales").limit(5))

Bronze Layer created


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


In [0]:
spark.table("default.bronze_sales").printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- Country: string (nullable = true)



**2. Data Cleaning & Transformation (Silver Layer)**


In [0]:
#Silver layer --- (Silver Customers)
from pyspark.sql.functions import col

# select 'CustomerId' and 'Country' columns and not take repeat rows and filtering null values
customers_df = (spark.table("default.bronze_sales")
                .select("CustomerID", "Country")
                .distinct()
                .filter(col("CustomerID").isNotNull()))

customers_df.write.format("delta").mode("overwrite").saveAsTable("default.silver_customers") #  -- Saving 

print("Customers table created!!!!") # -- chiking with print created or not

Customers table created!!!!


In [0]:
display(spark.table("default.silver_customers").limit(5))


CustomerID,Country
17548,United Kingdom
13767,United Kingdom
17377,United Kingdom
15922,United Kingdom
17873,United Kingdom


In [0]:
#Silver Products 
# Select StockCode, Description and UnitPrice columns and not take repeat rows and filtering null values
products_df = (spark.table("default.bronze_sales")
               .select("StockCode", "Description", "UnitPrice")
               .distinct()
               .filter(col("StockCode").isNotNull()))

#saving 
products_df.write.format("delta").mode("overwrite").saveAsTable("default.silver_products")

print("Table Products ready!!!!")

Table Products ready!!!!


In [0]:
display(spark.table("default.silver_products").limit(5))


StockCode,Description,UnitPrice
21731,RED TOADSTOOL LED NIGHT LIGHT,1.65
22086,PAPER CHAIN KIT 50'S CHRISTMAS,2.55
37370,RETRO COFFEE MUGS ASSORTED,1.06
21931,JUMBO STORAGE BAG SUKI,1.95
22961,JAM MAKING SET PRINTED,1.45


In [0]:
#Silver Orders 
# We leave everything, but we remove the rows without CustomerID (this is garbage for analytics)
orders_df = spark.table("default.bronze_sales").filter(col("CustomerID").isNotNull())

# Saving 
orders_df.write.format("delta").mode("overwrite").saveAsTable("default.silver_orders")

print("Tabel Orders ready!!!")
display(spark.table("default.silver_orders").limit(5))

Tabel Orders ready!!!


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


**Remove Duplicates & Handle Nulls**

In [0]:
#Remove Duplicates & Handle Nulls
raw_df = spark.table("workspace.default.bronze_sales")

cleaned_df = raw_df.dropna(subset=["CustomerID", "StockCode", "InvoiceNo"]).dropDuplicates()

print("1. Duplicates are deleted, empty values are processed!!!!")

1. Duplicates are deleted, empty values are processed


**Create Derived Columns**

In [0]:
#Create Derived Columns
from pyspark.sql.functions import col, round

# Adding the LineTotal column (Quantity * UnitPrice)
df_with_columns = cleaned_df.withColumn("LineTotal", round(col("Quantity") * col("UnitPrice"), 2))

print("2. Calculated Columns have been added!!!")


2. Calculated Columns have been added!!!


In [0]:
#Window Functions
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Creating a window: group by client and sort by date
windowSpec = Window.partitionBy("CustomerID").orderBy("InvoiceDate")

# We add the serial number of the purchase for each customer
df_final_silver = df_with_columns.withColumn("purchase_step", row_number().over(windowSpec))

print("3. Window Function применена (нумерация покупок).")

3. Window Function применена (нумерация покупок).


In [0]:
#delete the old tables to create them with a new structure
spark.sql("DROP TABLE IF EXISTS workspace.default.silver_customers")
spark.sql("DROP TABLE IF EXISTS workspace.default.silver_products")
spark.sql("DROP TABLE IF EXISTS workspace.default.silver_orders")

DataFrame[]

In [0]:
# Normalize Schema
# Saving Customers
(df_final_silver.select("CustomerID", "Country").distinct()
 .write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_customers"))

# Saving  Products
(df_final_silver.select("StockCode", "Description", "UnitPrice").distinct()
 .write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_products"))

# Saving Orders (this is our main table)
(df_final_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_orders"))

print("4. The data is normalized and stored in 3 Silver Layer tables!!!!")

4. The data is normalized and stored in 3 Silver Layer tables!!!!


In [0]:
#checking after do normalize schema droping tables 
display(spark.table("workspace.default.silver_orders").limit(10))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineTotal,purchase_step
541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1/18/2011 10:01,1.04,12346,United Kingdom,77183.6,1
C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1/18/2011 10:17,1.04,12346,United Kingdom,-77183.6,2
542237,84625A,PINK NEW BAROQUECANDLESTICK CANDLE,24,1/26/2011 14:30,0.85,12347,Iceland,20.4,1
542237,84625C,BLUE NEW BAROQUE CANDLESTICK CANDLE,24,1/26/2011 14:30,0.85,12347,Iceland,20.4,2
542237,85116,BLACK CANDELABRA T-LIGHT HOLDER,6,1/26/2011 14:30,2.1,12347,Iceland,12.6,3
542237,20719,WOODLAND CHARLOTTE BAG,10,1/26/2011 14:30,0.85,12347,Iceland,8.5,4
542237,22375,AIRLINE BAG VINTAGE JET SET BROWN,4,1/26/2011 14:30,4.25,12347,Iceland,17.0,5
542237,22376,AIRLINE BAG VINTAGE JET SET WHITE,4,1/26/2011 14:30,4.25,12347,Iceland,17.0,6
542237,20966,SANDWICH BATH SPONGE,10,1/26/2011 14:30,1.25,12347,Iceland,12.5,7
542237,22725,ALARM CLOCK BAKELIKE CHOCOLATE,4,1/26/2011 14:30,3.75,12347,Iceland,15.0,8


Gold Layer!!!!
Create business-level tables including Customers, Sales summary, and
Product performance.

In [0]:
#Here we will calculate how many orders each customer has made and how much money he has brought.
from pyspark.sql.functions import count, sum, col

# We count the activity of each client
gold_customers = (spark.table("workspace.default.silver_orders")
                  .groupBy("CustomerID", "Country")
                  .agg(
                      count("InvoiceNo").alias("TotalOrders"),
                      sum("LineTotal").alias("TotalSpent")
                  )
                  .orderBy(col("TotalSpent").desc()))

# Saving in  Gold
gold_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_customers_report")

print("1. Gold: The client report is ready!!!")
display(gold_customers.limit(5))

1. Gold: The client report is ready!!!


CustomerID,Country,TotalOrders,TotalSpent
14646,Netherlands,2085,279489.0200000003
18102,United Kingdom,433,256438.49000000002
17450,United Kingdom,350,187322.16999999998
14911,EIRE,5898,132458.7299999991
12415,Australia,778,123725.44999999985


In [0]:
#Sales Summary
#show the total revenue by country
gold_sales_summary = (spark.table("workspace.default.silver_orders")
                      .groupBy("Country")
                      .agg(sum("LineTotal").alias("TotalRevenue"))
                      .orderBy(col("TotalRevenue").desc()))

# Saving in  Gold
gold_sales_summary.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_sales_summary")

print("2. Gold: Done gold_sales_summary!!!")
display(gold_sales_summary.limit(5))

2. Gold: Done gold_sales_summary!!!


Country,TotalRevenue
United Kingdom,6747156.150003607
Netherlands,284661.5400000004
EIRE,250001.78000000003
Germany,221509.46999999814
France,196626.0500000001


Databricks visualization. Run in Databricks to view.

In [0]:
# Product Performance
# Product rating by the number of pieces sold
gold_product_performance = (spark.table("workspace.default.silver_orders")
                            .groupBy("StockCode")
                            .agg(
                                sum("Quantity").alias("UnitsSold"),
                                sum("LineTotal").alias("ProductRevenue")
                            )
                            .orderBy(col("UnitsSold").desc()))

# Saving в Gold
gold_product_performance.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_product_performance")

print("3. Gold: The product rating is ready!!!")
display(gold_product_performance.limit(5))

3. Gold: The product rating is ready!!!


StockCode,UnitsSold,ProductRevenue
84077,53119,13304.490000000025
22197,48689,36840.33000000012
85099B,44963,83056.5200000013
84879,35215,56331.90999999945
85123A,34185,93923.1499999993


Databricks visualization. Run in Databricks to view.

**Incremental Processing**
Implement incremental ingestion using MERGE INTO or CDC logic.

In [0]:
#this is a new_selver_data
new_silver_data = (spark.table("workspace.default.silver_orders")
                   .groupBy("CustomerID", "Country")
                   .agg(
                       count("InvoiceNo").alias("TotalOrders"),
                       sum("LineTotal").alias("TotalSpent")
                   ))
new_silver_data = new_silver_data.dropDuplicates(["CustomerID"]) # checking to duplicates fo Customer_ID
new_silver_data.createOrReplaceTempView("updates_view")

In [0]:
#do Merge Into 
#it's mean do SQL inside of Python (Change Data Capture)
spark.sql("""
MERGE INTO workspace.default.gold_customers_report AS target
USING updates_view AS source
ON target.CustomerID = source.CustomerID
WHEN MATCHED THEN
  UPDATE SET 
    target.TotalOrders = source.TotalOrders,
    target.TotalSpent = source.TotalSpent
WHEN NOT MATCHED THEN
  INSERT (CustomerID, Country, TotalOrders, TotalSpent) 
  VALUES (source.CustomerID, source.Country, source.TotalOrders, source.TotalSpent)
""")

print("✅ Incremental Load (MERGE) donee!")

✅ Incremental Load (MERGE) donee!


In [0]:
display(spark.table("workspace.default.gold_customers_report").limit(10)) #checking the result after incremental load

CustomerID,Country,TotalOrders,TotalSpent
12346,United Kingdom,2,0.0
12347,Iceland,182,4310.000000000001
12348,Finland,31,1797.2399999999998
12349,Italy,73,1757.55
12350,Norway,17,334.40000000000003
12352,Norway,95,1545.4099999999999
12353,Bahrain,4,89.0
12354,Spain,58,1079.3999999999999
12355,Bahrain,13,459.4
12356,Portugal,59,2811.4300000000007


**Data Quality Checks**_
_

In [0]:
#Checking for empty values (Null Checks)
null_count = spark.table("workspace.default.silver_orders").filter(col("CustomerID").isNotNull() == False).count()

if null_count == 0:
    print("✅ Verification passed: There are no empty values in the CustomerID column")
else:
    print(f"❌Attention! Found {null_count} empty values in CustomerID.")

✅ Verification passed: There are no empty values in the CustomerID column


In [0]:
#Adding Constraints (Constraints of the Delta Table)
#We add a restriction: the quantity of the product cannot be negative (if it is not a refund)
# We check the current data first, and then we set the rule
try:
    spark.sql("ALTER TABLE workspace.default.silver_orders ADD CONSTRAINT quantityPositive CHECK (Quantity > 0)")
    print("✅ Constraint added: It is now impossible to insert Quantity <= 0")
except:
    print("⚠️The constrain already exists or there are errors (returns) in the data.")

# Adding a check that the client's ID is not empty
try:
    spark.sql("ALTER TABLE workspace.default.silver_orders CHANGE COLUMN CustomerID SET NOT NULL")
    print("✅The NOT NULL constraint has been added for the CustomerID..")
except:
    print("⚠️Couldn't add NOT NULL. There may be empty entries in the table.")

⚠️The constrain already exists or there are errors (returns) in the data.
✅The NOT NULL constraint has been added for the CustomerID..


## # **Vizualization by Country and Total revenue **
![image_1776278868119.png](./image_1776278868119.png "image_1776278868119.png")

![image_1776278955153.png](./image_1776278955153.png "image_1776278955153.png")

### Data Governance & Catalog Structure
![image_1776278127882.png](./image_1776278127882.png "image_1776278127882.png")

## Conclusion

In this project, I successfully implemented a complete Medallion Architecture using Databricks and Delta Lake.

Key takeaways and business value:

- **Data Integrity:** By implementing the Silver Layer, I ensured that the data is clean, normalized, and free of duplicates or null values.
- **Business Insights:** The Gold Layer provides high-level aggregations that allow for immediate analysis of customer behavior and product performance. 

*This architecture transforms raw, messy sales data into a reliable "Single Source of Truth," enabling data-driven decision-making for the business.*